In [ ]:
!git clone https://github.com/IDKiro/DehazeFormer.git
%cd DehazeFormer
!pip install -r requirements.txt
!pip install pytorch-msssim --quiet

In [ ]:
import torch 
from models import dehazeformer_t, dehazeformer_s, dehazeformer_b, dehazeformer_m

device = 'cuda' if torch.cuda.is_available() else 'cpu'
path = "/kaggle/input/dehaze/dehazeformer-b.pth"

num_epochs = 10
batch_size = 8
grad_steps = 4
lr = 1e-4

model = dehazeformer_b().to(device)
weights = torch.load(path, map_location=device)["state_dict"]
state_dict = {k.replace("module.", ""): v for k, v in weights.items()}
model.load_state_dict(state_dict)

In [ ]:
from torch.utils.data import Subset, Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import os
import pandas as pd
import numpy as np

class DehazeDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        clear_path = self.data.iloc[idx]['GT']
        hazy_path = self.data.iloc[idx]['Hazy']

        # --- Load and convert to RGB ---
        clear = np.array(Image.open(clear_path).convert('RGB'), dtype=np.float32)
        hazy  = np.array(Image.open(hazy_path).convert('RGB'), dtype=np.float32)

        # --- Scale [0,255] -> [0,1] ---
        clear /= 255.0
        hazy  /= 255.0

        # --- Normalize [0,1] -> [-1,1] ---
        clear = clear * 2.0 - 1.0
        hazy  = hazy * 2.0 - 1.0

        # --- Apply albumentations transforms ---
        if self.transform:
            augmented = self.transform(image=hazy, image2=clear)
            hazy  = augmented["image"]
            clear = augmented["image2"]

        return clear, hazy

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

resize = (256, 256)

transform = A.Compose([
    A.Resize(height=resize[0], width=resize[1]),

    # Geometric
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, border_mode=0, p=0.3),
    A.Affine(
        scale=(0.98, 1.02),
        translate_percent=(0.02, 0.02),
        rotate=0,
        p=0.2
    ),

    # Appearance
    A.RandomBrightnessContrast(brightness_limit=0.05, contrast_limit=0.05, p=0.3),
    #A.RandomGamma(gamma_limit=(90, 110), p=0.2),

    # Noise & Blur
    A.MultiplicativeNoise(multiplier=(0.95, 1.05), per_channel=False, p=0.2),
    A.MedianBlur(blur_limit=3, p=0.1),

    # Local distortions
    A.CLAHE(clip_limit=(1.5, 2.5), tile_grid_size=(8, 8), p=0.2),
    A.GridDistortion(num_steps=5, distort_limit=0.02, p=0.1),

    # tensor
    ToTensorV2(),
], additional_targets={'image2': 'image'})
val_transform = A.Compose([
    A.Resize(height=resize[0], width=resize[1]),
    ToTensorV2(),
], additional_targets={'image2': 'image'})

In [ ]:
train = DehazeDataset('/kaggle/input/dehazing-dataset-thesis/dehaing_dataset_train.csv', transform=transform)
val = DehazeDataset('/kaggle/input/dehazing-dataset-thesis/dehazing_dataset_val.csv', transform=val_transform)
test = DehazeDataset('/kaggle/input/dehazing-dataset-thesis/dehazing_dataset_test.csv', transform=val_transform)
sample = DehazeDataset('/kaggle/input/dehazing-dataset-thesis/dehazing_dataset_sample_test.csv', transform=val_transform)

In [ ]:
print(len(train))           #  -> 14,135
print(len(val))             #  -> 20
print(len(test))            #  -> 520
print(len(sample))          #  -> 5

train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val, batch_size=4, shuffle=False)

In [ ]:
import matplotlib.pyplot as plt

clear, hazy = next(iter(val_loader))
print(clear.shape, hazy.shape)

# map [-1, 1] → [0, 1] for display
def denormalize(img_tensor):
    return (img_tensor * 0.5) + 0.5  # [-1,1] → [0,1]

for i in range(2):
    plt.figure(figsize=(14, 6))

    plt.subplot(1,3,1)
    plt.imshow(denormalize(clear[i]).permute(1,2,0).cpu().numpy())
    plt.title("Clear")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(denormalize(hazy[i]).permute(1,2,0).cpu().numpy())
    plt.title("Hazy")
    plt.axis("off")

    plt.show()

In [ ]:
from torch import nn, optim
from pytorch_msssim import MS_SSIM

# --- Loss Functions ---
criterion_l1 = nn.L1Loss()
criterion_ssim = MS_SSIM(data_range=1.0, size_average=True, channel=3)

# --- Optimizer & Scheduler ---
optimizer = optim.AdamW(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

In [ ]:
from torch.amp import autocast
from tqdm.notebook import tqdm
import torch.nn.functional as F

# --- Combined Criterion Function ---
def criterion(output, target):
    l1 = criterion_l1(output, target)
    ssim = criterion_ssim(output, target)

    return l1 + ssim

# --- Training Function ---
def train_epoch(train_loader, network, criterion, optimizer, scaler, grad_steps=1, device="cpu"):
    network.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)

    for step, (clear, hazy) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        clear, hazy = clear.to(device), hazy.to(device)

        with autocast(device):
            output = network(hazy)
            loss = criterion(output, clear)

        scaler.scale(loss).backward()

        if (step + 1) % grad_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item())

        del loss, output

    return sum(losses) / max(1, len(losses))


# --- Validation Function ---
def validate(val_loader, network, criterion, device):
    network.eval()
    losses = []
    
    with torch.no_grad():
        for clear, hazy in tqdm(val_loader, desc="Validation", leave=False):
            clear, hazy = clear.to(device), hazy.to(device)

            with autocast(device):
                output = network(hazy)
                loss = criterion(output, clear)

            losses.append(loss.item())

            del loss, output

    return sum(losses) / max(1, len(losses))


# --- PSNR Evaluation ---
def evaluate_psnr(test_loader, network, device="cpu"):
    network.eval()
    psnr_scores = []

    with torch.no_grad():
        for clear, hazy in tqdm(test_loader, desc="Testing", leave=False):
            clear, hazy = clear.unsqueeze(0).to(device), hazy.unsqueeze(0).to(device)

            with autocast(device):
                output = network(hazy)

            # Map [-1,1] -> [0,1] for PSNR computation
            output = denormalize(output)
            clear  = denormalize(clear)

            # Compute per-sample MSE
            mse = F.mse_loss(output, clear, reduction='none')
            mse = mse.mean(dim=[1, 2, 3])  # per-sample MSE

            # Compute PSNR per sample
            psnr = 10 * torch.log10(1.0 / mse)
            psnr_scores.extend(psnr.tolist())

            # free memory
            del output, mse, psnr

    avg_psnr = sum(psnr_scores) / max(1, len(psnr_scores))
    return avg_psnr


# --- Plot Curves ---
def plot(psnr, train, val):
    print(f"PSNR Score: {psnr:.2f}")
    plt.figure(figsize=(8, 5))
    plt.plot(train, label="Train Loss")
    plt.plot(val, label="Validation Loss")
    plt.title("Training vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
from torch.amp import GradScaler

scaler = GradScaler(device)
train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch [{epoch}/{num_epochs}]")
    train_loss = train_epoch(train_loader, model, criterion, optimizer, scaler, grad_steps, device)
    val_loss = validate(val_loader, model, criterion, device)
        
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    print(f"\t- Train Loss: {train_loss:.4f} | Validation Loss: {val_loss:.4f}")

psnr = evaluate_psnr(test, model, device)

# Save the model weights
torch.save(model.state_dict(), "/kaggle/working/dehazefomer.pth")

In [ ]:
plot(psnr, train_losses, val_losses)

In [ ]:
# Get a batch of clear and hazy images
clear, hazy = next(iter(val_loader))
clear, hazy = clear.to(device), hazy.to(device)

# Put model in eval mode
model.eval()
with torch.no_grad():
    pred = model(hazy)

# Visualize a few samples
for i in range(4):
    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(denormalize(clear[i]).permute(1, 2, 0).cpu().numpy())
    plt.title("Clear (Ground Truth)")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(denormalize(hazy[i]).permute(1, 2, 0).cpu().numpy())
    plt.title("Hazy (Input)")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(denormalize(pred[i]).permute(1, 2, 0).cpu().numpy())
    plt.title("Predicted (Dehazed)")
    plt.axis("off")

    plt.show()